In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 170)
np.set_printoptions(suppress=True)

In [3]:
import os
from google.colab import files
files.upload()

DATA, BOOK = 'knhanes_like_week4.csv', 'codebook_week4.csv'
df = pd.read_csv(DATA)
w  = df['wt'].to_numpy()
ST, PS = df['kstrata'].to_numpy(), df['psu'].to_numpy()

print(f'n = {len(df):,}  |  조사구 {df.psu.nunique()}개  |  층 {df.kstrata.nunique()}개')
print(f'가중치 합 = {w.sum():,.0f}')

Saving codebook_week4.csv to codebook_week4.csv
Saving knhanes_like_week4.csv to knhanes_like_week4.csv
n = 5,367  |  조사구 263개  |  층 18개
가중치 합 = 450,899


In [14]:
import numpy as np
import pandas as pd

# 1. 데이터 불러오기
df = pd.read_csv('knhanes_like_week4.csv')

# 2. 전처리 (무응답 코드 9 처리 및 노출/비노출 정의)
# edu: 1, 2 -> 저학력(노출군), 3, 4 -> 고졸 이상(비노출군), 9/NaN -> 결측 제외
df['exposure'] = np.nan
df.loc[df['edu'].isin([1, 2]), 'exposure'] = 1
df.loc[df['edu'].isin([3, 4]), 'exposure'] = 0

# income: 무응답(9)을 NaN 처리 및 소득 1분위(범주형 0/1) 변수 생성
df['income_clean'] = df['income'].replace(9, np.nan)
df['income_q1'] = np.where(df['income_clean'].isna(), np.nan, (df['income_clean'] == 1).astype(float))

# 범주형 분석 변수 정제 (0/1 binary indicator로 변환)
df['female'] = np.where(df['sex'].isna(), np.nan, (df['sex'] == 2).astype(float))
df['eupmyeon'] = np.where(df['urban'].isna(), np.nan, (df['urban'] == 2).astype(float))
df['smk_curr'] = np.where(df['smk'].isna(), np.nan, (df['smk'] == 1).astype(float))
df['drink_high'] = np.where(df['drink'].isna(), np.nan, (df['drink'] == 3).astype(float))
df['exercise_yes'] = np.where(df['exercise'].isna(), np.nan, (df['exercise'] == 2).astype(float))
df['htn_yes'] = np.where(df['htn'].isna(), np.nan, (df['htn'] == 1).astype(float))


# 3. 복합표본 도메인 분석 및 SMD 계산 함수
def svy_domain_analysis_with_smd(data, var_name, is_categorical=False):
    sub_df = data.dropna(subset=[var_name, 'exposure', 'wt', 'kstrata', 'psu']).copy()

    # 범주형 변수는 0~1 비율(proportion)로 계산 (표시할 때는 % 적용)
    sub_df['y'] = sub_df[var_name]

    # --- [전체 모집단 가중 추정치 & SE] ---
    w_tot = sub_df['wt'].sum()
    mean_tot = (sub_df['y'] * sub_df['wt']).sum() / w_tot

    sub_df['u_tot'] = (sub_df['wt'] / w_tot) * (sub_df['y'] - mean_tot)
    psu_tot = sub_df.groupby(['kstrata', 'psu'])['u_tot'].sum().reset_index()
    st_means_tot = psu_tot.groupby('kstrata')['u_tot'].mean().reset_index().rename(columns={'u_tot': 'st_mean'})
    psu_tot = psu_tot.merge(st_means_tot, on='kstrata')

    var_tot = 0
    for st, group in psu_tot.groupby('kstrata'):
        nh = len(group)
        if nh > 1:
            var_tot += (nh / (nh - 1)) * np.sum((group['u_tot'] - group['st_mean']) ** 2)
    se_tot = np.sqrt(var_tot)

    # --- [도메인별(노출군/비노출군) 가중 추정치 & SE & 분산] ---
    domain_res = {}
    for d_val in [1, 0]: # 1: 저학력, 0: 고졸 이상
        d_mask = (sub_df['exposure'] == d_val)
        w_d = sub_df.loc[d_mask, 'wt'].sum()
        mean_d = (sub_df.loc[d_mask, 'y'] * sub_df.loc[d_mask, 'wt']).sum() / w_d

        # 가중 분산(Weighted Variance) 계산 (SMD용)
        var_weighted_d = (sub_df.loc[d_mask, 'wt'] * (sub_df.loc[d_mask, 'y'] - mean_d)**2).sum() / w_d

        # Design-based SE
        sub_df['u_d'] = 0.0
        sub_df.loc[d_mask, 'u_d'] = (sub_df.loc[d_mask, 'wt'] / w_d) * (sub_df.loc[d_mask, 'y'] - mean_d)

        psu_d = sub_df.groupby(['kstrata', 'psu'])['u_d'].sum().reset_index()
        st_means_d = psu_d.groupby('kstrata')['u_d'].mean().reset_index().rename(columns={'u_d': 'st_mean'})
        psu_d = psu_d.merge(st_means_d, on='kstrata')

        var_d = 0
        for st, group in psu_d.groupby('kstrata'):
            nh = len(group)
            if nh > 1:
                var_d += (nh / (nh - 1)) * np.sum((group['u_d'] - group['st_mean']) ** 2)
        se_d = np.sqrt(var_d)

        domain_res[d_val] = {
            'mean': mean_d,
            'se': se_d,
            'var': var_weighted_d
        }

    # --- [SMD (Standardized Mean Difference) 계산] ---
    m1, m0 = domain_res[1]['mean'], domain_res[0]['mean']
    v1, v0 = domain_res[1]['var'], domain_res[0]['var']

    # Pooled Standard Deviation
    pooled_sd = np.sqrt((v1 + v0) / 2.0)
    smd = abs(m1 - m0) / pooled_sd if pooled_sd > 0 else 0.0

    # 범주형인 경우 % 표기를 위해 mean과 se에 100을 곱해줌
    mult = 100.0 if is_categorical else 1.0

    return {
        'total': {'w_mean': mean_tot * mult, 'se': se_tot * mult},
        'exp_1': {'w_mean': domain_res[1]['mean'] * mult, 'se': domain_res[1]['se'] * mult},
        'exp_0': {'w_mean': domain_res[0]['mean'] * mult, 'se': domain_res[0]['se'] * mult},
        'smd': smd
    }


# 4. 분석 변수 목록
var_list = [
    ('age', '연령 (세)', False),
    ('female', '여성 (%)', True),
    ('eupmyeon', '읍면 거주 (%)', True),
    ('income_q1', '소득 1분위 (%)', True),
    ('bmi', '체질량지수 (kg/m²)', False),
    ('sbp', '수축기 혈압 (mmHg)', False),
    ('dbp', '이완기 혈압 (mmHg)', False),
    ('chol', '총콜레스테롤 (mg/dL)', False),
    ('glucose', '공복혈당 (mg/dL)', False),
    ('smk_curr', '현재 흡연 (%)', True),
    ('drink_high', '고위험 음주 (%)', True),
    ('exercise_yes', '규칙적 운동 (%)', True),
    ('htn_yes', '고혈압 유병 (%)', True)
]

# 5. 테이블용 데이터 생성
rows_table1 = []
for var_col, var_label, is_cat in var_list:
    res = svy_domain_analysis_with_smd(df, var_col, is_categorical=is_cat)

    fmt_tot = f"{res['total']['w_mean']:.1f} ({res['total']['se']:.1f})"
    fmt_e1  = f"{res['exp_1']['w_mean']:.1f} ({res['exp_1']['se']:.1f})"
    fmt_e0  = f"{res['exp_0']['w_mean']:.1f} ({res['exp_0']['se']:.1f})"
    fmt_smd = f"{res['smd']:.3f}"

    rows_table1.append([var_label, fmt_tot, fmt_e1, fmt_e0, fmt_smd])

# N 수 계산
n_total = len(df.dropna(subset=['exposure']))
n_exp1  = (df['exposure'] == 1).sum()
n_exp0  = (df['exposure'] == 0).sum()

# Table 1 DataFrame 생성
df_table1_smd = pd.DataFrame(rows_table1, columns=[
    '특성 변수',
    f'전체 (N={n_total:,})',
    f'노출군: 저학력 (n={n_exp1:,})',
    f'비노출군: 고졸 이상 (n={n_exp0:,})',
    'SMD'

])

# 출력
display(df_table1_smd)

,특성 변수,"전체 (N=5,287)","노출군: 저학력 (n=1,761)","비노출군: 고졸 이상 (n=3,526)",SMD
0,연령 (세),49.1 (0.2),60.0 (0.4),45.3 (0.2),0.960
1,여성 (%),51.0 (0.7),51.9 (1.3),50.7 (0.9),0.025
2,읍면 거주 (%),23.0 (0.2),35.8 (0.9),18.5 (0.3),0.398
3,소득 1분위 (%),23.9 (0.6),44.7 (1.3),17.1 (0.7),0.626
4,체질량지수 (kg/m²),23.8 (0.1),24.1 (0.1),23.7 (0.1),0.107
5,수축기 혈압 (mmHg),122.8 (0.4),129.0 (0.6),120.6 (0.5),0.473
6,이완기 혈압 (mmHg),73.4 (0.3),75.8 (0.4),72.5 (0.3),0.310
7,총콜레스테롤 (mg/dL),170.4 (0.5),172.8 (0.9),169.5 (0.6),0.105
8,공복혈당 (mg/dL),91.6 (0.3),94.5 (0.5),90.7 (0.3),0.226
9,현재 흡연 (%),24.2 (0.7),24.6 (1.2),24.0 (0.8),0.014


<비가중 Table1>

In [19]:
import numpy as np
import pandas as pd

# 1. 데이터 불러오기
df = pd.read_csv('knhanes_like_week4.csv')

# 2. 전처리 (무응답 코드 9 처리 및 범주형 변수 정제)
df['exposure'] = np.nan
df.loc[df['edu'].isin([1, 2]), 'exposure'] = 1
df.loc[df['edu'].isin([3, 4]), 'exposure'] = 0

df['income_clean'] = df['income'].replace(9, np.nan)
df['income_q1'] = np.where(df['income_clean'].isna(), np.nan, (df['income_clean'] == 1).astype(float))

df['female'] = np.where(df['sex'].isna(), np.nan, (df['sex'] == 2).astype(float))
df['eupmyeon'] = np.where(df['urban'].isna(), np.nan, (df['urban'] == 2).astype(float))
df['smk_curr'] = np.where(df['smk'].isna(), np.nan, (df['smk'] == 1).astype(float))
df['drink_high'] = np.where(df['drink'].isna(), np.nan, (df['drink'] == 3).astype(float))
df['exercise_yes'] = np.where(df['exercise'].isna(), np.nan, (df['exercise'] == 2).astype(float))
df['htn_yes'] = np.where(df['htn'].isna(), np.nan, (df['htn'] == 1).astype(float))

# 3. 비가중 및 가중 추정치 비교 계산 함수
def compare_unweighted_weighted(data, var_name, is_categorical=False):
    # 해당 변수 및 가중치(wt) 결측 제외
    sub_df = data.dropna(subset=[var_name, 'wt']).copy()

    mult = 100.0 if is_categorical else 1.0

    # 1) 비가중 추정치 (단순 산술 평균/비율)
    unw_val = sub_df[var_name].mean() * mult

    # 2) 가중 추정치 (가중치 wt 적용 평균/비율)
    w_val = np.average(sub_df[var_name], weights=sub_df['wt']) * mult

    # 3) 차이 (비가중 - 가중)
    diff = unw_val - w_val

    return unw_val, w_val, diff

# 4. 분석할 변수 목록
var_list = [
    ('age', '연령 (세)', False),
    ('female', '여성 (%)', True),
    ('eupmyeon', '읍면 거주 (%)', True),
    ('income_q1', '소득 1분위 (%)', True),
    ('bmi', '체질량지수 (kg/m²)', False),
    ('sbp', '수축기 혈압 (mmHg)', False),
    ('dbp', '이완기 혈압 (mmHg)', False),
    ('chol', '총콜레스테롤 (mg/dL)', False),
    ('glucose', '공복혈당 (mg/dL)', False),
    ('smk_curr', '현재 흡연 (%)', True),
    ('drink_high', '고위험 음주 (%)', True),
    ('exercise_yes', '규칙적 운동 (%)', True),
    ('htn_yes', '고혈압 유병 (%)', True)
]

# 5. 비교 표 구축
rows_comparison = []

for var_col, var_label, is_cat in var_list:
    unw_v, w_v, diff_v = compare_unweighted_weighted(df, var_col, is_categorical=is_cat)

    unit = "%" if is_cat else ""
    diff_unit = "%p" if is_cat else ""

    rows_comparison.append([
        var_label,
        f"{unw_v:.1f}{unit}",
        f"{w_v:.1f}{unit}",
        f"{diff_v:+.1f}{diff_unit}"  # 부호(+/-) 명시
    ])

# DataFrame 생성 및 출력
df_comparison_table = pd.DataFrame(rows_comparison, columns=[
    '특성 변수',
    '비가중 추정치 (Unweighted)',
    '가중 추정치 (Weighted)',
    '격차 (비가중 - 가중)'
])

print("=============================")
print("    [비가중 vs 가중 비교] ")
print("=============================")
display(df_comparison_table)

    [비가중 vs 가중 비교] 


,특성 변수,비가중 추정치 (Unweighted),가중 추정치 (Weighted),격차 (비가중 - 가중)
0,연령 (세),54.0,49.1,+4.9
1,여성 (%),49.3%,51.0%,-1.6%p
2,읍면 거주 (%),40.2%,22.9%,+17.3%p
3,소득 1분위 (%),29.1%,23.9%,+5.2%p
4,체질량지수 (kg/m²),24.0,23.8,+0.2
5,수축기 혈압 (mmHg),125.7,122.8,+2.9
6,이완기 혈압 (mmHg),74.5,73.4,+1.1
7,총콜레스테롤 (mg/dL),172.0,170.4,+1.7
8,공복혈당 (mg/dL),92.8,91.6,+1.3
9,현재 흡연 (%),24.1%,24.1%,-0.0%p


## 각주 작성

1. n은 관측 수, %와 평균은 가중 추정치라는 것
2. 연속형 변수를 어떤 통계량으로 요약했는지
3. 집단 비교에 사용한 방법 (SMD 기준 포함)
4. 결측이 있었다면 그 처리 방식

> **각주:**

1.표에 제시된 n은 결측치를 제외한 단순 관측 수이며 백분율(%)과 평균은 복합표본 가중치, 층화변수, 조사구가 반영된 가중 추정치임.

2.연속형 변수는 가중 평균과 가중 표준오차(SE)로 요약하였음.

3.노출군(저학력)과 비노출군(고학력) 간의 집단 비교는 복합표본 도메인 분석과 표준화 평균 차이(SMD)를 활용하였으며 SMD 절댓값이 0.1 미만인 경우는 두 집단 간 불균형이 무시할 만한 차이로 판정함.

4.분석 과정에서 변수별로 결측값(학력 무응답 80명, 소득 무응답 326명 등)이 존재할 경우 해당 변수의 분석 대상에서 결측을 제외하여 처리함.